In [ ]:
# ✅ LSTM Hindi Poem Generator with Top-p Sampling
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dropout, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import random, nltk
nltk.download('punkt')

# Load data
with open("hindipoems.txt", encoding="utf-8") as f:
    data = f.read()

lines = [line.strip() for line in data.split("\n") if line.strip()]
tokenizer = Tokenizer()
tokenizer.fit_on_texts(lines)
total_words = len(tokenizer.word_index) + 1

# Prepare input sequences
input_sequences = []
for line in lines:
    tokens = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(tokens)):
        input_sequences.append(tokens[:i+1])

max_seq_len = max(len(seq) for seq in input_sequences)
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_seq_len, padding="pre"))

X, y = input_sequences[:,:-1], tf.keras.utils.to_categorical(input_sequences[:,-1], num_classes=total_words)

# Build model
model = Sequential([
    Embedding(total_words, 64, input_length=max_seq_len-1),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(64),
    Dense(total_words, activation="softmax")
])
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

# Train
model.fit(X, y, epochs=50, verbose=2)

# Save model
model.save("lstm_hindi_poem.h5")

# Top-p sampling generation
def top_p_sampling(preds, p=0.9):
    sorted_indices = np.argsort(preds)[::-1]
    cumulative = np.cumsum(preds[sorted_indices])
    cutoff = cumulative > p
    if np.any(cutoff):
        sorted_indices = sorted_indices[:np.argmax(cutoff)+1]
    sorted_probs = preds[sorted_indices]
    sorted_probs /= sorted_probs.sum()
    return np.random.choice(sorted_indices, p=sorted_probs)

def generate_poem(seed, num_words=20, top_p_value=0.9):
    for _ in range(num_words):
        token_list = tokenizer.texts_to_sequences([seed])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding="pre")
        preds = model.predict(token_list, verbose=0)[0]
        next_index = top_p_sampling(preds, p=top_p_value)
        next_word = tokenizer.index_word.get(next_index, "")
        seed += " " + next_word
    return seed

# Example
print("📜", generate_poem("प्रकृति की", 20))
